In [1]:
#THIS NOTEBOOK CONTAINS THE SENSITIVITY ANALYSIS USED TO INVESTIGATE THE EFFECT OF DIFFERENT SPLITTING TYPES, NUMBERS OF FOLDS, TRAINING METRICS AND DROP OUT REGULARIZATION 

#import sys
#sys.path.append("..")
from chemprop_utils import (                                    #IMPORTING RELEVANT FUNCTIONS FROM THE GENERAL FUNCTION SCRIPT
    load_and_clean_data, build_splits, make_kfold_splits,
    build_and_train_mpnn, predict_test_set,
)
from chemprop import nn                       #IMPORTS THE NEURAL NETWORK LIBRARY                      
from pathlib import Path                      #IMPORTS FILE PATH HANDLING LIBRARY
import pandas as pd                           #IMPORTS PANDAS FOR DATAFRAME HANDLING
import numpy as np                            #IMPORTS LIBRARY FOR BASIC CALCULATIONS
import torch                                  #IMPORTS TENSOR CALCULATOR
from lightning import pytorch as pl           #IMPORTS ADDITIONAL LIBRARIES USED BY GENERAL FUNCTIONS
#NOTE: THESE LIBRARIES ARE GENERALLY NOT USED DIRECTLY IN THIS SCRIPT, BUT ARE NECESSARY AS THEY ARE USED BYFUNCTIONS IN CHEMPROP_UTILS, WHICH ARE CALLED BY THIS SCRIPT

pl.seed_everything(0, workers=True)           #INITIALIZES THE SEED AND NUMBER OF WORKERS TO ZERO

input_path = Path("RAW DATA") / "dHfus_DB.csv"  #EXTRACTS RAW DATA USING THE FILE PATH AND DEFINES THE SMILES AND FUSION ENTAHLPY TARGET COLUMNS
smiles_column = "smiles"
target_columns = ["dHfus [kcal/mol]"]

df_clean, all_data, mols = load_and_clean_data(input_path, smiles_column, target_columns)   #CLEANS THE DATA AS DONE IN 0.1_DATA_VALIDATION USING THE GENERAL FUNCTION
splits = build_splits(all_data, mols)                                                       #SPLITS THE DATA BY CALLING THE DEFINED SPLITTING FUNCTION

results_log = []                 #MAKES AN EMPTY LIST TO BE USED FOR RESULT STORAGE. ONLY RUN ONCE PER SESSION TO AVOID RESETING ALL LOGGED RESULTS
predictions_log = []

#DEFINITION OF RESULTS-LOGGING FUNCTION. TAKES A RESULT AND RUN LABELLING INFORMATION FROM THE BUILD_AND_TRAIN_MPNN() FUNCTION 
#AND APPENDS ONE ROW SUMMARISING RUN RESULTS
def log_run(result, run_name, split_type=None, k=None, fold_idx=None, metric_used=None,
            dropout=None, n_train=None, n_val=None, n_test=None):
    tm = result["test_results"][0]                                          #EXTRACTS THE AGGREGATE TEST METRICS DICTIONARY
    row = {
        "run_name": run_name, "split_type": split_type, "k": k, "fold_idx": fold_idx,
        "metric_used": metric_used, "dropout": dropout,
        "rmse": tm.get("test/rmse"), "mae": tm.get("test/mae"), "mse": tm.get("test/mse"),   
        "n_train": n_train, "n_val": n_val, "n_test": n_test,
    }
    results_log.append(row)                                                 #ADDS THIS RUN'S ROW TO THE RUNNING LIST OF ALL LOGGED RESULTS
    return row                                                              #RETURNS THE ROW DIRECTLY FOR INSPECTING A SINGLE RUN'S RESULT WITHOUT PULLING THE FULL LOG

def results_to_df():
    return pd.DataFrame(results_log)              #FORMATS RESULTS INTO A READABLE AND VIEWABLE DATAFRAME


#BELOW ARE THE ACTUAL IMPLEMENTATIONS OF THE SENSITIVITY ANALYSIS
#ALL FOUR TESTS FOLLOW THE SAME STRUCTURE
# STEP 1: THE BUILD_AND_TAIN_MPNN FUNCTION IS CALLED USING SPECIFIED CONDITIONS
# STEP 2: THE RESULTS ARE LOGGED
# STEP 3: PREDICTIONS ARE COLLECTED EXPLICTLY IF REQUIRED
# STEP 4: PROGRESS IS SAVED AND THE RUN STATUS IS PRINTED
#NOTE: IF THE METRIC, K NUMBER AND DROPOUT ARE NOT SPECIFIED, THEY ASSUME THEIR DEFINED DEFAULT VALUES FROM THE FUNCTIONS SCRIPT

# 1. SCAFFOLD VS RANDOM SPLITTING

for split_name, s in splits.items():
    run_name = f"dHfus_{split_name}"
    result = build_and_train_mpnn(s["train"], s["val"], s["test"], run_name=run_name)
    log_run(result, run_name, split_type=split_name,
            n_train=len(s["train"][0]), n_val=len(s["val"][0]), n_test=len(s["test"][0]))
    predictions_log.append(predict_test_set(result, run_name))
    results_to_df().to_csv("sensitivity_results.csv", index=False)
    print(f"Done: {run_name}")


# 2. K-FOLD 

kfold_pools = {1: [splits["scaffold_balanced"]]}
for k in [3, 5, 10]:                                            #3, 5 AND 10 FOLDS ARE TESTED AS THEY ARE THE CONVENTION ML CHOICES
    kfold_pools[k] = make_kfold_splits(all_data, k=k, seed=0)   

for k, folds in kfold_pools.items():
    for i, f in enumerate(folds):
        run_name = f"dHfus_k{k}_fold{i}"
        result = build_and_train_mpnn(f["train"], f["val"], f["test"], run_name=run_name)
        log_run(result, run_name, k=k, fold_idx=i,
                n_train=len(f["train"][0]), n_val=len(f["val"][0]), n_test=len(f["test"][0]))
        results_to_df().to_csv("sensitivity_results.csv", index=False)
        print(f"Done: {run_name}")


# 3. TRAINING METRIC COMPARISON

metric_variants = {
    "mse": [nn.metrics.MSE(), nn.metrics.MAE(), nn.metrics.RMSE()],
    "mae": [nn.metrics.MAE(), nn.metrics.MSE(), nn.metrics.RMSE()],
    "rmse": [nn.metrics.RMSE(), nn.metrics.MAE(), nn.metrics.MSE()],
}
for metric_name, metric_list in metric_variants.items():
    run_name = f"dHfus_metric_{metric_name}"
    s = splits["scaffold_balanced"]
    result = build_and_train_mpnn(s["train"], s["val"], s["test"], metric_list=metric_list, run_name=run_name)
    log_run(result, run_name, split_type="scaffold_balanced", metric_used=metric_name,
            n_train=len(s["train"][0]), n_val=len(s["val"][0]), n_test=len(s["test"][0]))
    predictions_log.append(predict_test_set(result, run_name))
    results_to_df().to_csv("sensitivity_results.csv", index=False)
    print(f"Done: {run_name}")


# 4. DROPOUT COMPARISON

for dropout_val in [0.0, 0.1, 0.2, 0.4]:                    #THE BASELINE USES NO DROPOUT. 0.1 AND 0.2 ARE COMMON AND 0.4 IS AGRESSIVE
    run_name = f"dHfus_dropout_{dropout_val}"
    s = splits["scaffold_balanced"]
    result = build_and_train_mpnn(s["train"], s["val"], s["test"], dropout=dropout_val, run_name=run_name)
    log_run(result, run_name, split_type="scaffold_balanced", dropout=dropout_val,
            n_train=len(s["train"][0]), n_val=len(s["val"][0]), n_test=len(s["test"][0]))
    predictions_log.append(predict_test_set(result, run_name))
    results_to_df().to_csv("sensitivity_results.csv", index=False)
    print(f"Done: {run_name}")

results_to_df()

Seed set to 0


Loaded 5184 rows, dropped 527 with missing target, 4657 remain


[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] WARNING: not removing hydrogen atom without neighbors
[16:00:41] Unusual charge on atom 6 number of radical electrons set to zero
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\astartes\samplers\extrapolation\scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 1308 molecules corresponding to indices {3, 6, 9, 12, 13, 22, 29, 39, 41, 50, 56, 62, 63, 70, 71, 83, 85, 86, 88, 90, 95, 97, 99, 106, 109, 118, 121, 129, 132, 133, 135, 138, 148, 149, 150, 152

scaffold_balanced: train=3727, val=465, test=465
random: train=3725, val=465, test=467


c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

... (320 outputs left out)

Output()

c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Done: dHfus_dropout_0.4


,run_name,split_type,k,fold_idx,metric_used,dropout,rmse,mae,mse,n_train,n_val,n_test
0,dHfus_scaffold_balanced,scaffold_balanced,NaN,NaN,NaN,NaN,2.508873,1.746371,NaN,3727,465,465
1,dHfus_random,random,NaN,NaN,NaN,NaN,3.992173,1.957838,NaN,3725,465,467
2,dHfus_k1_fold0,NaN,1.0,0.0,NaN,NaN,2.284226,1.610408,NaN,3727,465,465
3,dHfus_k3_fold0,NaN,3.0,0.0,NaN,NaN,3.844981,2.086210,NaN,2794,310,1553
4,dHfus_k3_fold1,NaN,3.0,1.0,NaN,NaN,3.149158,1.781645,NaN,2795,310,1552
5,dHfus_k3_fold2,NaN,3.0,2.0,NaN,NaN,3.965247,1.880686,NaN,2795,310,1552
6,dHfus_k5_fold0,NaN,5.0,0.0,NaN,NaN,3.717584,1.971144,NaN,3353,372,932
7,dHfus_k5_fold1,NaN,5.0,1.0,NaN,NaN,3.503380,2.082276,NaN,3353,372,932
8,dHfus_k5_fold2,NaN,5.0,2.0,NaN,NaN,3.838225,2.123035,NaN,3354,372,931
9,dHfus_k5_fold3,NaN,5.0,3.0,NaN,NaN,3.641784,1.777204,NaN,3354,372,931
